In [1]:
import pandas as pd
train_files = ['/kaggle/input/acne-severity-classification/Classification/NNEW_trainval_0.txt','/kaggle/input/acne-severity-classification/Classification/NNEW_trainval_1.txt',
        '/kaggle/input/acne-severity-classification/Classification/NNEW_trainval_2.txt','/kaggle/input/acne-severity-classification/Classification/NNEW_trainval_3.txt',
        '/kaggle/input/acne-severity-classification/Classification/NNEW_trainval_4.txt']

test_files = ['/kaggle/input/acne-severity-classification/Classification/NNEW_test_0.txt','/kaggle/input/acne-severity-classification/Classification/NNEW_test_1.txt',
        '/kaggle/input/acne-severity-classification/Classification/NNEW_test_2.txt','/kaggle/input/acne-severity-classification/Classification/NNEW_test_3.txt',
        '/kaggle/input/acne-severity-classification/Classification/NNEW_test_4.txt']
path = '/kaggle/input/acne-severity-classification/Classification/JPEGImagesv'

In [2]:
from PIL import Image
import os
import torch
from sklearn.model_selection import train_test_split

    
# define a data class
class ClassificationDataset:
    def __init__(self, data, data_path, transform, training=True):
        """Define the dataset for classification problems

        Args:
            data ([dataframe]): [a dataframe that contain 2 columns: image name and label]
            data_path ([str]): [path/to/folder that contains image file]
            transform : [augmentation methods and transformation of images]
            training (bool, optional): []. Defaults to True.
        """
        self.data = data
        self.imgs = data["path"].unique().tolist()
        self.data_path = data_path
        self.training = training
        self.transform = transform

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.data_path, self.data.iloc[idx, 0]))

        if(self.training):
            label = self.data.iloc[idx, 1]
        if self.transform is not None:
            img = self.transform(img)
        if(self.training):
            return img, label
        else:
            return img

    def __len__(self):
        return len(self.imgs)


def make_loader(dataset, train_batch_size, validation_split=0.2):
    """make dataloader for pytorch training

    Args:
        dataset ([object]): [the dataset object]
        train_batch_size ([int]): [training batch size]
        validation_split (float, optional): [validation ratio]. Defaults to 0.2.

    Returns:
        [type]: [description]
    """
    # number of samples in train and test set
    train_len = int(len(dataset) * (1 - validation_split))
    test_len = len(dataset) - train_len
    train_set, test_set = torch.utils.data.random_split(dataset, [train_len, test_len])
    # create train_loader
    print(len(train_set))
    train_loader = torch.utils.data.DataLoader(
        train_set, batch_size=train_batch_size, shuffle=True,
    )
    # create test_loader
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=1, shuffle=False,)
    return train_loader, test_loader


def data_split(data, test_size):
    x_train, x_test, y_train, y_test = train_test_split(
        data, data["label"], test_size=test_size, stratify = data.iloc[:,1]
    )
    return x_train, x_test, y_train, y_test

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
# config
bs = 16

In [4]:
from sklearn import metrics as skmetrics
import numpy
class Metrics:
    def __init__(self, metric_names):
        self.metric_names = metric_names
        # initialize a metric dictionary
        self.metric_dict = {metric_name: [0] for metric_name in self.metric_names}

    def step(self, labels, preds):
        for metric in self.metric_names:
            # get the metric function
            do_metric = getattr(
                skmetrics, metric, "The metric {} is not implemented".format(metric)
            )
            # check if metric require average method, if yes set to 'micro' or 'macro' or 'None'
            try:
                self.metric_dict[metric].append(
                    do_metric(labels, preds, average="macro")
                )
            except:
                self.metric_dict[metric].append(do_metric(labels, preds))

    def epoch(self):
        # calculate metrics for an entire epoch
        avg = [sum(metric) / (len(metric) - 1) for metric in self.metric_dict.values()]
        metric_as_dict = dict(zip(self.metric_names, avg))
        return metric_as_dict

    def last_step_metrics(self):
        # return metrics of last steps
        values = [self.metric_dict[metric][-1] for metric in self.metric_names]
        metric_as_dict = dict(zip(self.metric_names, values))
        return metric_as_dict

In [5]:
import torch.nn.functional as F
import torch.nn as nn

class LabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing=0.1, dim=-1):
        super(LabelSmoothingLoss, self).__init__()
        self.smoothing = smoothing
        self.dim = dim

    def forward(self, pred, target):
        target = F.one_hot(target, num_classes=pred.size(-1))
        target = target.float()
        target = (1 - self.smoothing) * target + self.smoothing / pred.size(-1)
        log_pred = F.log_softmax(pred, dim=self.dim)
        loss = nn.KLDivLoss(reduction='batchmean')(log_pred, target)
        return loss
    
criterion = LabelSmoothingLoss(smoothing=0.12)

In [6]:
def train_one_epoch(
    model,
    train_loader,
    test_loader,
    device,
    optimizer,
    criterion,
    train_metrics,
    val_metrics,
):

    # training-the-model
    train_loss = 0
    valid_loss = 0
    all_labels = []
    all_preds = []
    model.train()
    for data, target in train_loader:
        # move-tensors-to-GPU
        data = data.type(torch.FloatTensor).to(device)
#         day = day.view(-1,1).type(torch.FloatTensor).to(device)
        # target=torch.Tensor(target)
        target = target.float().to(device)
        # clear-the-gradients-of-all-optimized-variables
        optimizer.zero_grad()
        # forward-pass: compute-predicted-outputs-by-passing-inputs-to-the-model
        output = model(data)
        preds = torch.argmax(output, axis=1).cpu().detach().numpy()
        labels = target.cpu().numpy()
        # calculate-the-batch-loss
        loss = criterion(output.type(torch.FloatTensor), target.type(torch.LongTensor))
        # backward-pass: compute-gradient-of-the-loss-wrt-model-parameters
        loss.backward()
        # perform-a-ingle-optimization-step (parameter-update)
        optimizer.step()
        # update-training-loss
        train_loss += loss.item() * data.size(0)
        # calculate training metrics
        all_labels.extend(labels)
        all_preds.extend(preds)
    
    train_metrics.step(all_labels, all_preds)

    # validate-the-model
    model.eval()
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for data, target in test_loader:
            data = data.type(torch.FloatTensor).to(device)
#             day = day.view(-1,1).type(torch.FloatTensor).to(device)
            target = target.to(device)
            output = model(data)
            preds = torch.argmax(output, axis=1).tolist()
            labels = target.tolist()
            all_labels.extend(labels)
            all_preds.extend(preds)
            loss = criterion(output, target)

            # update-average-validation-loss
            valid_loss += loss.item() * data.size(0)

    val_metrics.step(all_labels, all_preds)
    train_loss = train_loss / len(train_loader.sampler)
    valid_loss = valid_loss / len(test_loader.sampler)

    return (
        train_loss,
        valid_loss,
        train_metrics.last_step_metrics(),
        val_metrics.last_step_metrics(),
    )

# Fold 0

In [7]:
train_df = pd.read_csv(train_files[0],names=['path','label','leisons'],sep='  ')

/tmp/ipykernel_22/2083062489.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  train_df = pd.read_csv(train_files[0],names=['path','label','leisons'],sep='  ')


In [8]:
train_df

,path,label,leisons
0,levle0_0.jpg,0,3
1,levle0_1.jpg,0,3
2,levle0_100.jpg,0,2
3,levle0_101.jpg,0,1
4,levle0_102.jpg,0,3
...,...,...,...
1160,levle3_92.jpg,3,63
1161,levle3_93.jpg,3,61
1162,levle3_94.jpg,3,58
1163,levle3_95.jpg,3,58


In [9]:
x_train, x_val, y_train, y_val = data_split(train_df,0.2)

In [10]:
test_df = pd.read_csv(test_files[0],names=['path','label','leisons'],sep='  ')

/tmp/ipykernel_22/2331413007.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  test_df = pd.read_csv(test_files[0],names=['path','label','leisons'],sep='  ')


In [11]:
test_df

,path,label,leisons
0,levle0_451.jpg,0,2
1,levle0_498.jpg,0,1
2,levle0_485.jpg,0,3
3,levle0_218.jpg,0,2
4,levle1_344.jpg,0,4
...,...,...,...
287,levle3_88.jpg,3,64
288,levle3_1.jpg,3,60
289,levle3_68.jpg,3,56
290,levle3_16.jpg,3,65


In [12]:
import torchvision
mean = (0.5, 0.5, 0.5)
std = (0.5, 0.5, 0.5)
transform = torchvision.transforms.Compose([torchvision.transforms.Resize((224, 224)),
                                            torchvision.transforms.RandomHorizontalFlip(p=0.5),  # Randomly flip images left-right
                            torchvision.transforms.RandomVerticalFlip(p=0.5),
                            torchvision.transforms.RandomRotation(degrees=15),
                                            torchvision.transforms.ElasticTransform(),
                                               torchvision.transforms.ToTensor(),
                                               torchvision.transforms.Normalize(mean, std)])
test_transform = torchvision.transforms.Compose([torchvision.transforms.Resize((224, 224)),
                                               torchvision.transforms.ToTensor(),
                                               torchvision.transforms.Normalize(mean, std)])

In [13]:
train_dataset = ClassificationDataset(x_train,data_path = "/kaggle/input/acne-severity-classification/Classification/JPEGImages",transform=transform,training=True)
val_dataset = ClassificationDataset(x_val,data_path = "/kaggle/input/acne-severity-classification/Classification/JPEGImages",transform=test_transform,training=True)
train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=bs, shuffle=True,
    )
    # create test_loader
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=False)

In [14]:
testset = ClassificationDataset(test_df,data_path = "/kaggle/input/acne-severity-classification/Classification/JPEGImages",transform=test_transform,training=True)
test_loader = torch.utils.data.DataLoader(
        testset, batch_size=1, shuffle=False,
    )

In [15]:
train_metrics = Metrics(["accuracy_score","f1_score"])
val_metrics = Metrics(["accuracy_score","f1_score"])

In [16]:
class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
     
        self.cnn = torchvision.models.efficientnet_v2_m(pretrained=True).cuda()
        for param in self.cnn.parameters():
            param.requires_grad = True
        self.cnn.classifier = nn.Sequential(
      
        nn.Linear(self.cnn.classifier[1].in_features, 512),
        nn.Dropout(p=0.2),
        nn.ReLU(),
        nn.Linear(512, 128),
        nn.Dropout(p=0.2),
        nn.Linear(128, 64),
            nn.Linear(64, 4),     
       )
        
    def forward(self, img):
        output = self.cnn(img)
        return output

In [17]:
model = MyNet().cuda()

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_M_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_M_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 200MB/s]


In [18]:
a= torch.ones((16,3,224,224)).cuda()

model(a)

tensor([[-0.0897,  0.1340,  0.1002, -0.1689],
        [-0.0947,  0.0949,  0.0163, -0.1798],
        [-0.0942,  0.1335,  0.0806, -0.1759],
        [-0.1282,  0.1024,  0.0533, -0.1836],
        [-0.1067,  0.0962,  0.0450, -0.1902],
        [-0.0865,  0.0909,  0.0438, -0.1929],
        [-0.0739,  0.1276,  0.0611, -0.2413],
        [-0.1437,  0.1046,  0.0837, -0.1372],
        [-0.0865,  0.1573,  0.0674, -0.1863],
        [-0.1140,  0.0903,  0.0627, -0.1297],
        [-0.0979,  0.1040,  0.0878, -0.1574],
        [-0.1028,  0.0951,  0.0653, -0.1839],
        [-0.1005,  0.0828,  0.0695, -0.1411],
        [-0.0810,  0.1030,  0.0645, -0.1845],
        [-0.1157,  0.0837,  0.0601, -0.1855],
        [-0.1082,  0.0935,  0.0482, -0.2119]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

In [19]:
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "min", patience=4, factor=0.5
    )

In [20]:
device = torch.device("cuda")

In [21]:
from tqdm import tqdm
for param in model.parameters():
    param.requires_grad = True

model = model.to(device)
num_epoch = 20
best_val_acc = 0.0
import logging
import numpy as np
print("begin training process")
for i in tqdm(range(0, num_epoch)):
    loss, val_loss, train_result, val_result = train_one_epoch(
        model,
        train_loader,
        val_loader,
        device,
        optimizer,
        criterion,
        train_metrics,
        val_metrics,
    )

    scheduler.step(val_loss)
#     scheduler.step()
    print(
        "Epoch {} / {} \n Training loss: {} - Other training metrics: ".format(
            i + 1, num_epoch, loss
        )
    )
    print(train_result)
    print(
        " \n Validation loss : {} - Other validation metrics:".format(val_loss)
    )
    print(val_result)
    print("\n")
    # saving epoch with best validation accuracy
    if (loss<0.04):
        # no saving
        continue
    if best_val_acc < float(val_result["accuracy_score"]):
        print(
            "Validation accuracy= "+
            str(val_result["accuracy_score"])+
            "===> Save best epoch"
        )
        best_val_acc = val_result["accuracy_score"]
        torch.save(
            model,
            "./" +  "best.pt"
        )
    else:
        print(
            "Validation accuracy= "+ str(val_result["accuracy_score"])+ "===> No saving"
        )
        continue

begin training process


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1 / 20 
 Training loss: 0.7893756671524866 - Other training metrics: 
{'accuracy_score': 0.4570815450643777, 'f1_score': 0.23193287444302552}
 
 Validation loss : 0.8132582521745575 - Other validation metrics:
{'accuracy_score': 0.5064377682403434, 'f1_score': 0.3546306343294525}


Validation accuracy= 0.5064377682403434===> Save best epoch


  5%|▌         | 1/20 [03:14<1:01:37, 194.63s/it]

Epoch 2 / 20 
 Training loss: 0.6065644294407234 - Other training metrics: 
{'accuracy_score': 0.6030042918454935, 'f1_score': 0.514928829131595}
 
 Validation loss : 0.5378853355418459 - Other validation metrics:
{'accuracy_score': 0.6781115879828327, 'f1_score': 0.5829192999840164}


Validation accuracy= 0.6781115879828327===> Save best epoch


...

Epoch 18 / 20 
 Training loss: 0.08837646997409829 - Other training metrics: 
{'accuracy_score': 0.9560085836909872, 'f1_score': 0.9629333114093456}
 
 Validation loss : 0.4182923171557326 - Other validation metrics:
{'accuracy_score': 0.7939914163090128, 'f1_score': 0.7934010840108401}


Validation accuracy= 0.7939914163090128===> No saving


 95%|█████████▌| 19/20 [56:22<02:56, 176.59s/it]

Epoch 19 / 20 
 Training loss: 0.08274621412541733 - Other training metrics: 
{'accuracy_score': 0.9613733905579399, 'f1_score': 0.964734458457706}
 
 Validation loss : 0.43305870316634876 - Other validation metrics:
{'accuracy_score': 0.7553648068669528, 'f1_score': 0.7343699839486357}


Validation accuracy= 0.7553648068669528===> No saving


100%|██████████| 20/20 [59:19<00:00, 177.96s/it]

Epoch 20 / 20 
 Training loss: 0.05603807697736143 - Other training metrics: 
{'accuracy_score': 0.9774678111587983, 'f1_score': 0.9815554691812017}
 
 Validation loss : 0.38470799693350116 - Other validation metrics:
{'accuracy_score': 0.7939914163090128, 'f1_score': 0.7898071314080902}


Validation accuracy= 0.7939914163090128===> No saving


## Fold 0 : Testing

In [22]:
def test_result(model, test_loader, device,name='no_tta_prob.npy'):
    # testing the model by turning model "Eval" mode
    model.eval()
    preds = []
    aprobs = []
    labels = []
    with torch.no_grad():
        for data,target in test_loader:
            # move-tensors-to-GPU
            data = data.to(device)
            # forward-pass: compute-predicted-outputs-by-passing-inputs-to-the-model
            output = model(data)
            prob = nn.Softmax(dim=1)
            # applying Softmax to results
            probs = prob(output)
            aprobs.append(probs.cpu())
            labels.append(target.cpu().numpy())
            preds.extend(torch.argmax(probs, axis=1).tolist())
    aprobs = np.array(aprobs)
    np.save(name,aprobs)
    return preds, np.array(labels)

In [23]:
test_model = torch.load("/kaggle/working/best.pt")
test_model = test_model.to(device)

In [24]:
preds,labels =test_result(test_model, test_loader, device)


/tmp/ipykernel_22/938738078.py:19: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  aprobs = np.array(aprobs)
/tmp/ipykernel_22/938738078.py:19: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  aprobs = np.array(aprobs)


In [25]:
from sklearn.metrics import classification_report as rp
print(rp(labels,preds))

              precision    recall  f1-score   support

           0       0.85      0.66      0.74       103
           1       0.68      0.87      0.77       127
           2       0.62      0.44      0.52        36
           3       0.87      0.77      0.82        26

    accuracy                           0.74       292
   macro avg       0.75      0.69      0.71       292
weighted avg       0.75      0.74      0.73       292

